# Baseline

Wir bauen vier regelbasierte Baselines — ohne Machine Learning. Jede berechnet den **mittleren `arrival_delay`** auf einer anderen Granularitätsstufe aus den Trainingsdaten und wendet diese Vorhersage auf den Testdatensatz an.

Ziel: Den **härtesten Gegner** identifizieren, den das ML-Modell schlagen muss.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import polars as pl
import numpy as np

TRAIN, TEST, lf = setup_analysis("06_prediction_1-baseline")

## Data

Wir verwenden `train_features.parquet` — das vollständige Feature-Set inkl. Rohspalten. Filter identisch mit `lf_clean` aus den Analysis-Notebooks.

In [ ]:
def apply_clean_filter(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Standard lf_clean filters — same as in all analysis notebooks."""
    return (
        lf
        .filter(pl.col("canceled") == False)
        .filter(
            ~(
                (pl.col("operating_date").dt.year() == 2025)
                & (pl.col("operating_date").dt.month() >= 11)
            )
        )
        .filter(pl.col("line_name") != "E")
        .filter(pl.col("stop_sequence") > 1)
    )


train_lf = apply_clean_filter(pl.scan_parquet(TRAIN))
test_lf  = apply_clean_filter(pl.scan_parquet(TEST))

# Collect only what we need for baseline computation
COLS_NEEDED = ["arrival_delay", "hour", "line_name", "stop_name"]

train = train_lf.select(COLS_NEEDED).collect()
test  = test_lf.select(COLS_NEEDED).collect()

print(f"Train rows: {len(train):,}")
print(f"Test rows:  {len(test):,}")

## Helpers

In [ ]:
def mae(actual: pl.Series, predicted: pl.Series) -> float:
    """Mean Absolute Error in seconds."""
    return (actual - predicted).abs().mean()


def rmse(actual: pl.Series, predicted: pl.Series) -> float:
    """Root Mean Squared Error in seconds."""
    return ((actual - predicted) ** 2).mean() ** 0.5


def otp_accuracy(actual: pl.Series, predicted: pl.Series, threshold: int = 60) -> float:
    """Share of predictions within ±threshold seconds of actual delay."""
    return ((actual - predicted).abs() <= threshold).mean()

## Baseline 1 — Grand Mean

Einfachste Baseline: Wir sagen für jede Fahrt **denselben Wert** vorher — den mittleren `arrival_delay` über alle Trainingsdaten.

In [ ]:
grand_mean = train["arrival_delay"].mean()
print(f"Grand mean delay: {grand_mean:.1f}s")

pred_grand = pl.Series("pred", [grand_mean] * len(test))

b1_mae  = mae(test["arrival_delay"], pred_grand)
b1_rmse = rmse(test["arrival_delay"], pred_grand)
b1_otp  = otp_accuracy(test["arrival_delay"], pred_grand)

print(f"MAE:  {b1_mae:.1f}s")
print(f"RMSE: {b1_rmse:.1f}s")
print(f"OTP accuracy (±60s): {b1_otp:.1%}")

## Baseline 2 — Hour Mean

Wir sagen den mittleren Delay **pro Stunde** vorher. Berücksichtigt Rush-Hour-Muster.

In [ ]:
hour_means = (
    train
    .group_by("hour")
    .agg(pl.col("arrival_delay").mean().alias("pred_hour_mean"))
)

test_b2 = test.join(hour_means, on="hour", how="left")
# Fallback for unseen hours (shouldn't happen, but safe)
test_b2 = test_b2.with_columns(
    pl.col("pred_hour_mean").fill_null(grand_mean)
)

b2_mae  = mae(test_b2["arrival_delay"], test_b2["pred_hour_mean"])
b2_rmse = rmse(test_b2["arrival_delay"], test_b2["pred_hour_mean"])
b2_otp  = otp_accuracy(test_b2["arrival_delay"], test_b2["pred_hour_mean"])

print(f"MAE:  {b2_mae:.1f}s")
print(f"RMSE: {b2_rmse:.1f}s")
print(f"OTP accuracy (±60s): {b2_otp:.1%}")

## Baseline 3 — Line Mean

Wir sagen den mittleren Delay **pro Linie** vorher. Berücksichtigt linienspezifische Charakteristiken.

In [ ]:
line_means = (
    train
    .group_by("line_name")
    .agg(pl.col("arrival_delay").mean().alias("pred_line_mean"))
)

test_b3 = test.join(line_means, on="line_name", how="left")
test_b3 = test_b3.with_columns(
    pl.col("pred_line_mean").fill_null(grand_mean)
)

b3_mae  = mae(test_b3["arrival_delay"], test_b3["pred_line_mean"])
b3_rmse = rmse(test_b3["arrival_delay"], test_b3["pred_line_mean"])
b3_otp  = otp_accuracy(test_b3["arrival_delay"], test_b3["pred_line_mean"])

print(f"MAE:  {b3_mae:.1f}s")
print(f"RMSE: {b3_rmse:.1f}s")
print(f"OTP accuracy (±60s): {b3_otp:.1%}")

## Baseline 4 — Stop Mean

Die härteste Baseline: Wir sagen den mittleren Delay **pro Haltestelle** vorher. Haltestellen haben sehr unterschiedliche strukturelle Delay-Profile — dieser Ansatz kommt dem ML-Modell am nächsten.

In [ ]:
stop_means = (
    train
    .group_by("stop_name")
    .agg(pl.col("arrival_delay").mean().alias("pred_stop_mean"))
)

test_b4 = test.join(stop_means, on="stop_name", how="left")
test_b4 = test_b4.with_columns(
    pl.col("pred_stop_mean").fill_null(grand_mean)
)

b4_mae  = mae(test_b4["arrival_delay"], test_b4["pred_stop_mean"])
b4_rmse = rmse(test_b4["arrival_delay"], test_b4["pred_stop_mean"])
b4_otp  = otp_accuracy(test_b4["arrival_delay"], test_b4["pred_stop_mean"])

print(f"MAE:  {b4_mae:.1f}s")
print(f"RMSE: {b4_rmse:.1f}s")
print(f"OTP accuracy (±60s): {b4_otp:.1%}")

## Ergebnis — Baseline Vergleich

In [ ]:
results = pl.DataFrame({
    "Baseline":     ["Grand Mean", "Hour Mean", "Line Mean", "Stop Mean"],
    "Granularität": ["—",          "Stunde",    "Linie",     "Haltestelle"],
    "MAE (s)":      [round(b1_mae, 1), round(b2_mae, 1), round(b3_mae, 1), round(b4_mae, 1)],
    "RMSE (s)":     [round(b1_rmse, 1), round(b2_rmse, 1), round(b3_rmse, 1), round(b4_rmse, 1)],
    "OTP ±60s":     [
        f"{b1_otp:.1%}",
        f"{b2_otp:.1%}",
        f"{b3_otp:.1%}",
        f"{b4_otp:.1%}",
    ],
})

show_df(results.to_pandas())

## Benchmark

Die **Stop Mean Baseline** ist unser Benchmark — sie ist die härteste regelbasierte Linie, die das LightGBM-Modell schlagen muss.

Das ML-Modell bringt gegenüber dieser Baseline dann Mehrwert, wenn es zusätzlich:
- Tageszeit und Wochentag berücksichtigt (Rush-Hour-Effekte)
- Wetterbedingungen einbezieht (Regen, Schnee)
- Event-Cluster erkennt
- Kombinationen dieser Faktoren lernt (Interaktionen)

**Ziel für das Modell:** MAE unter dem Stop-Mean-Wert.

In [ ]:
print(f"Benchmark (Stop Mean MAE): {b4_mae:.1f}s")
print(f"Das LightGBM-Modell muss MAE < {b4_mae:.1f}s erreichen, um die Baseline zu schlagen.")